# 02 — Nettoyage et construction des indicateurs

Objectif : passer du format long (une ligne par agrégat) au format large (une ligne par commune/année), filtrer sur le périmètre d'étude, et construire les indicateurs financiers manquants (taux d'endettement, ratio investissement).

In [1]:
import sys
sys.path.append('../src')

import pandas as pd
from utils import load_raw_data, clean_columns, pivot_agregats, compute_taux_endettement

In [3]:
df = load_raw_data('../data/raw/communes.csv')
df.shape

(2755101, 42)

## Filtrage sur le budget principal

Le dataset contient budgets principaux et annexes. On ne garde que le budget principal pour ne pas fausser les agrégats globaux.

In [ ]:
df['Type de budget'].value_counts()

In [ ]:
df_bp = df[df['Type de budget'] == 'Budget principal'].copy()
df_bp.shape

## Pivot : format long -> format large

Une ligne par commune/année, une colonne par agrégat (Epargne brute, Encours de dette, etc.).

In [ ]:
pivot = pivot_agregats(df_bp)
pivot.shape

In [ ]:
pivot.head()

## Sélection des colonnes utiles

On garde uniquement ce qui sert à répondre aux 4 questions métier du README.

In [ ]:
colonnes_utiles = [
    'Exercice', 'Nom 2025 Commune', 'Nom 2025 Département', 'Strate population 2025',
    'Epargne brute', 'Encours de dette', 'Recettes de fonctionnement',
    "Dépenses d'investissement", 'Recettes totales'
]
pivot = pivot[colonnes_utiles]
pivot.head()

## Construction des indicateurs manquants

In [ ]:
pivot = compute_taux_endettement(pivot)
pivot['ratio_investissement'] = pivot["Dépenses d'investissement"] / pivot['Recettes totales']
pivot.head()

## Valeurs manquantes et doublons

In [ ]:
pivot.isna().mean()

In [ ]:
print('Doublons :', pivot.duplicated().sum())

## Décision sur les valeurs manquantes

À ajuster selon ce que renvoie `.isna().mean()` ci-dessus. Ne pas faire un dropna() aveugle sur tout le DataFrame — cibler les colonnes critiques pour l'analyse.

In [ ]:
# pivot = pivot.dropna(subset=['Epargne brute', 'Encours de dette', 'Recettes de fonctionnement'])
# pivot.shape

## Sauvegarde du dataset nettoyé

In [ ]:
pivot.to_csv('../data/raw/communes_occitanie_clean.csv', index=False)